# Sesión 4 · Redes y datos en Azure
**Machine Learning y Seguridad en la Nube — Especialización en Big Data (UCompensar)**

Este notebook acompaña las [diapositivas de la sesión](https://jotamao1985.github.io/Machine-Learning-cloud-computing_Ucompensar/diapositivas/sesion-4-redes-y-datos-en-azure.html).
Cada celda lleva entre corchetes el mismo identificador que el código proyectado, así que puedes seguir
la clase celda a celda. La portada del curso está en https://jotamao1985.github.io/Machine-Learning-cloud-computing_Ucompensar/
y el enunciado del proyecto final, en [su página](https://jotamao1985.github.io/Machine-Learning-cloud-computing_Ucompensar/proyecto.html).

1. **Archivo → Guardar una copia en Drive** para que tus cambios no se pierdan.
2. Ejecuta las celdas **en orden** con **Mayús + Enter**. Varias usan resultados de las anteriores.
   Si Colab avisa que el notebook no es de Google, elige **Ejecutar de todos modos**.
3. Si Colab se reinicia o una celda falla al repetirla, usa **Entorno de ejecución → Ejecutar anteriores**.
   Las celdas de ejercicio dan error mientras tengan `...` sin reemplazar (si el error menciona `Ellipsis`
   o `ellipsis`, te quedó un `...`). Colab se detiene en el primer error: completa los ejercicios en orden,
   o ejecuta con **Mayús + Enter** solo las celdas que necesites.
4. La primera celda **descarga** los datos del caso *Yarumo Fibra S.A.S.*, una empresa **ficticia** (sus
   datos se generaron para el curso y no corresponden a personas ni a empresas reales), y comprueba que
   el archivo sea idéntico al del curso. Si la descarga falla, la celda te dice cómo subirlo a mano.
5. Hoy entrenamos redes neuronales con **Keras**, que Colab ya trae. Basta el entorno normal, sin GPU: el
   notebook entero tarda unos minutos. Las redes empiezan con pesos al azar; aunque fijamos la semilla,
   en otro equipo (o con GPU) las cifras pueden salir un poco distintas de las de las diapositivas. Los
   verificadores lo tienen en cuenta.

En el código, los decimales van con punto (así los escribe Python); en el texto, con coma.

In [ ]:
#@title ▶ Descarga los datos del caso (ejecútala sin modificar) { display-mode: "form" }
"""Datos del caso «Yarumo Fibra S.A.S.» (empresa FICTICIA), descargados del repositorio del curso.

Deja en datos/clientes.csv los 8000 clientes del curso y comprueba su huella SHA-256: si el
archivo no es idéntico al del curso, no se usa. Solo usa la biblioteca estándar de Python.
"""

import hashlib
import os
import urllib.request
from pathlib import Path

DATOS_URL = "https://raw.githubusercontent.com/JotaMao1985/Machine-Learning-cloud-computing_Ucompensar/main/datos/clientes.csv"
DATOS_SHA256 = "2d4fad61a49efcf1d071b6c412e5224e71eca23047cbbac6b4017c8627f9c93b"
PORTADA = "https://jotamao1985.github.io/Machine-Learning-cloud-computing_Ucompensar/"


def _es_del_curso(contenido):
    return hashlib.sha256(contenido).hexdigest() == DATOS_SHA256


def _sin_datos(motivo):
    return (f"No pude dejar listos los datos: {motivo}.\n"
            f"Descarga clientes.csv desde la portada del curso ({PORTADA}), súbelo a Colab con el "
            "ícono de carpeta de la izquierda (Subir al almacenamiento de la sesión) y vuelve a "
            "ejecutar esta celda.")


def _leer_url():
    # Solo al preparar el material: la variable apunta a una copia local del mismo archivo
    local = os.environ.get("YARUMO_CSV_LOCAL")
    if local:
        return Path(local).read_bytes()
    with urllib.request.urlopen(DATOS_URL, timeout=30) as respuesta:
        return respuesta.read()


def descargar_datos(carpeta="datos"):
    """Deja <carpeta>/clientes.csv, idéntico al del curso, y devuelve su ruta."""
    ruta = Path(carpeta) / "clientes.csv"
    subido = Path("clientes.csv")  # por si se subió a mano a la carpeta de trabajo de Colab
    if ruta.exists() and _es_del_curso(ruta.read_bytes()):
        contenido, origen = ruta.read_bytes(), "ya estaban"
    elif subido.exists() and _es_del_curso(subido.read_bytes()):
        contenido, origen = subido.read_bytes(), "copiados del archivo que subiste"
    else:
        try:
            contenido = _leer_url()
        except OSError as error:
            raise RuntimeError(_sin_datos(f"no se pudo descargar el archivo ({error})")) from None
        if not _es_del_curso(contenido):
            raise RuntimeError(_sin_datos("el archivo descargado no es el del curso"))
        origen = "descargados del repositorio del curso"
    if origen != "ya estaban":
        ruta.parent.mkdir(parents=True, exist_ok=True)
        temporal = ruta.with_suffix(".tmp")  # sin archivos a medias si algo falla al escribir
        temporal.write_bytes(contenido)
        temporal.replace(ruta)
    lineas = contenido.decode("utf-8").splitlines()
    print(f"Listo: {ruta} con {len(lineas) - 1} clientes y {len(lineas[0].split(','))} columnas, {origen}")
    return ruta


"""Ayudas comunes a todos los notebooks del curso (van en la celda oculta, tras descarga_datos.py).

Las de cada sesión están en ayudas_sesionN.py.
"""


def _aviso_version():
    """Las diapositivas se ejecutaron con pandas 2.2 y scikit-learn 1.6: si Colab trae otras, se dice."""
    import pandas as pd
    import sklearn

    if pd.__version__.split(".")[:2] != ["2", "2"] or sklearn.__version__.split(".")[:2] != ["1", "6"]:
        print(f"Nota: este Colab trae pandas {pd.__version__} y scikit-learn {sklearn.__version__}; "
              "las diapositivas se hicieron con pandas 2.2 y scikit-learn 1.6. Si alguna salida "
              "se ve un poco distinta, es por eso.")


_aviso_version()


"""Sesión 4: estilo de las gráficas del curso y los verificadores de la práctica.

Van dentro de la celda oculta del notebook, junto con descarga_datos.py y ayudas.py. Cada
verificador recibe lo que el estudiante calculó, dice qué está bien y, en lo demás, da una
pista sin la respuesta. Nunca se cae: un «...» sin reemplazar o un tipo inesperado se avisa.

Keras se importa de forma perezosa: este archivo no lo importa (lo importa el estudiante en la
celda [keras], después de fijar el backend). Los verificadores de Keras NO reentrenan: usan la
red que el estudiante entrenó (solo predicen) y comparan lo que entregó con lo que esa red da.
Como Keras puede variar entre equipos, lo que se exige de la red son rangos, que la celda
[semillas] de celdas_sesion4.py comprueba con otras semillas al construir el material:

- ejercicio 1 (scikit-learn, determinista): los F1 del MLP de 16 y 8 neuronas, sin escalar y con
  escalado, calculados aquí con la misma semilla de la clase; tolerancia ±0,005 con escalado y ±0,02
  sin escalar (todos los F1 sin escalar son casi 0); las pistas exactas (±0,001) se prueban antes;
- ejercicio 2: el F1 de prueba de la red del estudiante en _RANGO_F1_RED y su sensibilidad en
  _RANGO_SENS_RED (rangos del barrido de 24 semillas del revisor 4: ninguna de 264 redes razonables
  queda fuera); lo entregado debe coincidir (±0,006, para aceptar el redondeo a dos decimales) con lo
  que da su propia red; se mira el historial (si paró, con qué paciencia, si volvió a los mejores
  pesos, comparando la pérdida de la red en la validación con la mínima y con la última del historial);
- ejercicio 3: con sus propias redes, la exactitud normal ≥ _MINIMO_NORMAL y, con las imágenes
  corridas un píxel, la CNN por encima de la densa en más de _MARGEN_CORRIDA.
"""

_RANGO_F1_RED = (0.38, 0.62)
_RANGO_SENS_RED = (0.25, 0.65)
_MINIMO_NORMAL = 0.93
_MARGEN_CORRIDA = 0.15


def _estilo_graficas():
    """Colores del curso y letras que se leen en Colab y proyectadas."""
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    plt.rcParams.update({
        "axes.prop_cycle": mpl.cycler(color=["#8A2E9E", "#F46201", "#566B7F"]),
        "figure.figsize": (6.4, 3.6),
        "figure.constrained_layout.use": True,
        "font.size": 13,
        "axes.titlesize": 15,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#94A3B8",
        "axes.labelcolor": "#334155",
        "xtick.color": "#475569",
        "ytick.color": "#475569",
        "text.color": "#1E293B",
    })


_estilo_graficas()

_CACHE = {}   # lo que ya se calculó: los datos preparados y los modelos de referencia


def _clientes():
    """Las particiones de la clase (la de la sesión 2 y la validación de Keras), o None con aviso."""
    if "clientes" in _CACHE:
        return _CACHE["clientes"]
    import pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    try:
        clientes = pd.read_csv("datos/clientes.csv")
    except FileNotFoundError:
        print("✗  No encuentro datos/clientes.csv: ejecuta primero la celda que descarga los datos "
              "(Entorno de ejecución → Ejecutar anteriores)")
        return None
    Xc = pd.get_dummies(clientes.drop(columns=["id_cliente", "cancelo"]), drop_first=True, dtype=int)
    yc = (clientes["cancelo"] == "Sí").astype(int)
    Xc_e, Xc_p, yc_e, yc_p = train_test_split(Xc, yc, test_size=0.25, random_state=2026, stratify=yc)
    Xc_e2, Xc_v, yc_e2, yc_v = train_test_split(Xc_e, yc_e, test_size=0.2, random_state=2026, stratify=yc_e)
    escala = StandardScaler().fit(Xc_e2)
    _CACHE["clientes"] = {
        "Xc_e": Xc_e, "Xc_p": Xc_p, "yc_e": yc_e, "yc_p": yc_p,
        "Xr_e": escala.transform(Xc_e2).astype("float32"), "Xr_v": escala.transform(Xc_v).astype("float32"),
        "Xr_p": escala.transform(Xc_p).astype("float32"),
        "yr_e": yc_e2.to_numpy(), "yr_v": yc_v.to_numpy(), "yr_p": yc_p.to_numpy(),
    }
    return _CACHE["clientes"]


def _digitos():
    """Las imágenes de la clase: entrenamiento y prueba, como en la celda [digitos]."""
    if "digitos" in _CACHE:
        return _CACHE["digitos"]
    from sklearn.datasets import load_digits
    from sklearn.model_selection import train_test_split

    d = load_digits()
    imagenes = (d.images / 16).astype("float32")[..., None]
    I_e, I_p, d_e, d_p = train_test_split(imagenes, d.target, test_size=0.25, random_state=2026,
                                          stratify=d.target)
    _CACHE["digitos"] = {"I_e": I_e, "I_p": I_p, "d_e": d_e, "d_p": d_p}
    return _CACHE["digitos"]


def _sobrantes(sueltos, otros, nombres, ejemplo):
    """Avisa de valores sin nombre o con otro nombre. True si hubo alguno."""
    if sueltos:
        print(f"✗  Pasa los valores por nombre, como en la plantilla: {ejemplo}")
        return True
    if otros:
        print(f"✗  No conozco {', '.join(repr(k) for k in otros)}: usa estos nombres, en "
              f"minúsculas y sin tildes: {', '.join(n + '=' for n in nombres)}")
        return True
    return False


def _seguro(verificador):
    """Red de seguridad: si algo imprevisto falla al revisar, se avisa sin traceback."""
    import functools

    @functools.wraps(verificador)
    def envoltura(*args, **kwargs):
        try:
            return verificador(*args, **kwargs)
        except Exception as error:   # noqa: BLE001 — el estudiante nunca ve un traceback
            print(f"✗  No pude revisar lo que entregaste ({type(error).__name__}). Revisa que cada valor "
                  "sea el resultado de un cálculo (un número) y vuelve a ejecutar la celda")
    return envoltura


def _un_numero(x):
    """(valor, None) si x es un número (o algo con un solo número); si no, (None, motivo)."""
    import math
    import numbers
    import numpy as np
    import pandas as pd

    if x is Ellipsis:
        return None, "todavía tiene los puntos suspensivos (...) de la plantilla: reemplázalos por tu cálculo"
    if isinstance(x, pd.DataFrame):
        if x.size != 1:
            return None, f"es una tabla de {x.shape[0]} × {x.shape[1]}; hace falta un solo número"
        x = x.iloc[0, 0]
    if isinstance(x, (pd.Series, pd.Index, np.ndarray, list, tuple)) or (
            hasattr(x, "shape") and hasattr(x, "numpy") and getattr(x, "shape", ()) != ()):
        arreglo = np.asarray(x, dtype=object)
        valores = list(np.ravel(arreglo))
        if len(valores) == 0:
            return None, "está vacío"
        if len(valores) > 1:
            if len(valores) in (450, 2000, 4800, 6000):
                return None, ("es una predicción por cliente o por imagen; falta resumirla en un número "
                              "(una métrica)")
            return None, f"tiene {len(valores)} valores; hace falta uno solo"
        x = valores[0]
        if x is Ellipsis:
            return None, "todavía tiene los puntos suspensivos (...) de la plantilla: reemplázalos por tu cálculo"
    if hasattr(x, "numpy") and not isinstance(x, (numbers.Real, np.number)):
        try:
            x = x.numpy()   # un tensor de un solo número
        except Exception:   # noqa: BLE001
            pass
    if x is None:
        return None, "está vacío (None): ¿le asignaste el resultado de un print?"
    if callable(x):
        return None, "es una función o un modelo, no su resultado: ¿faltó calcular la métrica?"
    if isinstance(x, (bool, np.bool_)):
        return None, "es True o False; hace falta un número"
    if isinstance(x, str):
        return None, "es texto; escríbelo como número, sin comillas y con punto decimal"
    if not isinstance(x, (numbers.Real, np.number)) or isinstance(x, np.complexfloating):
        return None, f"no es un número real (es {type(x).__name__})"
    try:
        valor = float(x)
    except (OverflowError, TypeError, ValueError):
        return None, "no es un número que se pueda revisar"
    if math.isnan(valor):
        return None, "salió nan: revisa el cálculo"
    if math.isinf(valor):
        return None, "salió infinito: ¿dividiste entre cero?"
    return valor, None


def _cerca(a, b, tolerancia):
    return a is not None and b is not None and abs(a - b) <= tolerancia


def _coma(x, decimales=3):
    """Un número con coma decimal, para los mensajes."""
    return f"{x:.{decimales}f}".replace(".", ",").replace("-", "−")


def _leer(entregados, nombres):
    """Convierte cada valor entregado en número; imprime los problemas. Devuelve {nombre: valor}."""
    valores, problemas = {}, {}
    for nombre in nombres:
        x = entregados[nombre]
        if x is Ellipsis:
            print(f"·  {nombre}: todavía tiene los puntos suspensivos (...) de la plantilla: reemplázalos por tu cálculo")
            continue
        valor, problema = _un_numero(x)
        if problema:
            problemas.setdefault(problema, []).append(nombre)
        elif not 0 <= valor <= 1:
            if 1 < valor <= 100:
                problemas.setdefault("es una proporción entre 0 y 1, no un porcentaje", []).append(nombre)
            else:
                problemas.setdefault("estas métricas van de 0 a 1", []).append(nombre)
        else:
            valores[nombre] = valor
    for problema, quienes in problemas.items():   # la misma pista una sola vez
        print(f"✗  {', '.join(quienes)}: {problema}")
    return valores


def _es_red(x):
    """¿Es un modelo de Keras (algo con capas y predict)?"""
    return x is not Ellipsis and hasattr(x, "predict") and hasattr(x, "layers")


def _capas(red):
    return [type(c).__name__ for c in red.layers]


def _llamadas_para(riesgo, y_real, meta=0.70):
    """Llamadas, de mayor a menor riesgo, para encontrar a la meta (la función de la clase)."""
    import numpy as np
    for corte in np.sort(np.unique(riesgo))[::-1]:
        llamados = riesgo >= corte
        if y_real[llamados].sum() >= meta * y_real.sum():
            return int(llamados.sum())


# ---------------------------------------------------------------------------------------------
# Ejercicio 1 · Escalar importa (scikit-learn: se puede calcular aquí, en una fracción de segundo)
# ---------------------------------------------------------------------------------------------

def _referencia_escalado():
    """F1, exactitud y sensibilidad de los MLP de la clase, sin escalar y con escalado."""
    if "escalado" in _CACHE:
        return _CACHE["escalado"]
    import warnings
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
    from sklearn.neural_network import MLPClassifier
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    d = _clientes()
    ref = {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        for capas in [(16, 8), (32,), (16,), (8,)]:
            for escalado in (False, True):
                mlp = MLPClassifier(hidden_layer_sizes=capas, early_stopping=True, random_state=2026)
                modelo = make_pipeline(StandardScaler(), mlp) if escalado else mlp
                modelo.fit(d["Xc_e"], d["yc_e"])
                pp, pe = modelo.predict(d["Xc_p"]), modelo.predict(d["Xc_e"])
                ref[(capas, escalado)] = {
                    "f1": f1_score(d["yc_p"], pp, zero_division=0),
                    "exactitud": accuracy_score(d["yc_p"], pp),
                    "sensibilidad": recall_score(d["yc_p"], pp, zero_division=0),
                    "precision": precision_score(d["yc_p"], pp, zero_division=0),
                    "f1_entrena": f1_score(d["yc_e"], pe, zero_division=0)}
    _CACHE["escalado"] = ref
    return ref


@_seguro
def revisar_escalado(*sueltos, f1_sin=..., f1_con=..., **otros):
    """Ejercicio 1: el F1 en prueba del MLP de 16 y 8 neuronas, sin escalar y con escalado."""
    nombres = ["f1_sin", "f1_con"]
    if _sobrantes(sueltos, otros, nombres, "revisar_escalado(f1_sin=f1_sin, f1_con=f1_con)"):
        return
    if _clientes() is None:
        return
    vals = _leer({"f1_sin": f1_sin, "f1_con": f1_con}, nombres)
    if not vals:
        return
    ref = _referencia_escalado()
    tol = 0.02
    bien = {"f1_sin": ref[((16, 8), False)], "f1_con": ref[((16, 8), True)]}
    if ("f1_sin" in vals and "f1_con" in vals and _cerca(vals["f1_sin"], bien["f1_con"]["f1"], tol)
            and _cerca(vals["f1_con"], bien["f1_sin"]["f1"], tol)
            and not _cerca(vals["f1_sin"], bien["f1_sin"]["f1"], tol)):
        print("✗  f1_sin y f1_con están cruzados: el de mlp_sin va en f1_sin y el de mlp_con, en f1_con")
        return
    ok = {}
    for nombre in nombres:
        if nombre not in vals:
            continue
        v, r = vals[nombre], bien[nombre]
        escalado = nombre == "f1_con"
        exacto = _cerca(v, r["f1"], 0.001 if escalado else 0.02)
        pista = None
        if exacto:
            pass
        elif escalado and _cerca(v, bien["f1_sin"]["f1"], 0.005):
            pista = ("salió como el del modelo sin escalar: ¿pusiste StandardScaler() como primer paso del "
                     "make_pipeline y mediste con mlp_con? Recuerda la SVM de la sesión 2 sin escalar")
        elif _cerca(v, r["exactitud"], 0.001):
            pista = "esa es la exactitud; aquí va el F1: f1_score(yc_prueba, …)"
        elif _cerca(v, r["sensibilidad"], 0.001):
            pista = "esa es la sensibilidad; aquí va el F1: f1_score(yc_prueba, …)"
        elif _cerca(v, r["precision"], 0.001):
            pista = "esa es la precisión; aquí va el F1: f1_score(yc_prueba, …)"
        elif _cerca(v, r["f1_entrena"], 0.001):
            pista = "ese es el F1 en el entrenamiento; se mide con la prueba (Xc_prueba y yc_prueba)"
        else:
            for capas in [(32,), (16,), (8,)]:
                if _cerca(v, ref[(capas, escalado)]["f1"], 0.001):
                    pista = ("es el de otra red: deja hidden_layer_sizes=(16, 8), dos capas de 16 y 8 neuronas")
                    break
        ok[nombre] = pista is None and _cerca(v, r["f1"], 0.005 if escalado else 0.02)
        if ok[nombre]:
            print(f"✓  {nombre}: bien")
            continue
        print(f"✗  {nombre}: {pista or 'no coincide; revisa que entrenes con Xc_ent, midas el F1 con Xc_prueba y dejes random_state=2026'}")
    if ok.get("f1_sin") and ok.get("f1_con"):
        s, c = bien["f1_sin"], bien["f1_con"]
        print(f"\n¡Listo! Sin escalar, el MLP tiene un F1 de {_coma(s['f1'])} y encuentra al "
              f"{_coma(s['sensibilidad'] * 100, 1)} % de los que cancelan; con escalado, el F1 sube a "
              f"{_coma(c['f1'])}. Explica en «Tus respuestas» por qué.")


# ---------------------------------------------------------------------------------------------
# Ejercicio 2 · Que no memorice (Keras: se usa la red del estudiante, sin reentrenar)
# ---------------------------------------------------------------------------------------------

def _entropia(y, p):
    """Entropía cruzada binaria media, como la calcula Keras (con el mismo recorte)."""
    import numpy as np
    p = np.clip(np.asarray(p, dtype="float64").ravel(), 1e-7, 1 - 1e-7)
    y = np.asarray(y, dtype="float64").ravel()
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())


@_seguro
def revisar_red(*sueltos, red=..., historial=..., f1=..., sensibilidad=..., **otros):
    """Ejercicio 2: la red con dropout y parada temprana; su F1 y su sensibilidad en la prueba."""
    import numpy as np
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

    nombres = ["red", "historial", "f1", "sensibilidad"]
    if _sobrantes(sueltos, otros, nombres, "revisar_red(red=red, historial=h, f1=f1_red, "
                                           "sensibilidad=sens_red)"):
        return
    if not _es_red(red):
        print("✗  red: pasa la red de Keras que entrenaste: red=red")
        return
    vl = None
    if isinstance(historial, dict):
        hist, epocas_max = historial, None
    elif hasattr(historial, "history") and isinstance(historial.history, dict):
        hist, epocas_max = historial.history, getattr(historial, "params", {}).get("epochs")
    else:
        print("✗  historial: pasa lo que devuelve fit, el historial del entrenamiento: historial=h")
        return
    if "val_loss" not in hist:
        print("✗  historial: no tiene pérdida de validación; entrena con validation_data=(Xr_val, yr_val)")
        return
    vl = list(hist["val_loss"])
    d = _clientes()
    if d is None:
        return
    try:
        prob = np.asarray(red.predict(d["Xr_p"], verbose=0)).ravel()
        prob_v = np.asarray(red.predict(d["Xr_v"], verbose=0)).ravel()
    except Exception as error:   # noqa: BLE001
        print(f"✗  red: no pude usarla con los clientes de prueba ({type(error).__name__}): ¿la entrada es "
              "keras.Input(shape=(Xr_ent.shape[1],))?")
        return
    if prob.size != len(d["yr_p"]):
        print("✗  red: debe dar una sola probabilidad por cliente: la última capa debe ser Dense(1, activation=\"sigmoid\")")
        return
    capas = _capas(red)
    tasas = [float(c.rate) for c in red.layers if type(c).__name__ == "Dropout"]
    # 1. ¿Hay dropout y parada temprana?
    if not tasas:
        print("·  La red no tiene capas Dropout: usa la arquitectura de la plantilla")
    elif max(tasas) == 0:
        print("·  Con una tasa de dropout de 0, no se apaga ninguna neurona: prueba entre 0.2 y 0.5 (con punto, como en el código)")
    elif max(tasas) >= 0.8:
        print(f"·  Con una tasa de dropout de {_coma(max(tasas), 2)} se apaga casi toda la red en cada paso: prueba entre 0.2 "
              "y 0.5 (con punto, como en el código)")
    n, mejor = len(vl), int(np.argmin(vl))
    paro = epocas_max is None or n < epocas_max
    if not paro:
        print(f"·  La red no paró: corrió las {n} épocas. Con restore_best_weights=True vuelve igual a su mejor "
              f"época (la {mejor + 1}), pero esperó de más: baja la paciencia")
    elif n - 1 - mejor <= 1 and n > 2:
        print("·  Paró en la primera época que no mejoró (paciencia 0 o 1): el ruido de la validación puede "
              "engañarla; prueba con una paciencia de 3 a 15")
    # 2. ¿Volvió a los mejores pesos? Con restore_best_weights=True, la pérdida de la red en la validación
    # es la mínima del historial (difieren ~1e-8); sin restaurar, es la de la última época.
    actual_v = _entropia(d["yr_v"], prob_v)
    d_min, d_ult = abs(actual_v - min(vl)), abs(actual_v - vl[-1])
    restaurada = d_min <= d_ult or vl[-1] - min(vl) < 1e-3
    if min(d_min, d_ult) > 5e-3:
        restaurada = False
        print("·  El historial no coincide con esta red en la validación: ¿es el h de esta misma red, entrenada "
              "con validation_data=(Xr_val, yr_val)?")
    elif not restaurada:
        print(f"✗  Tu red se quedó con los pesos de la última época (pérdida de validación {_coma(vl[-1])}) y no "
              f"con los de la mejor ({_coma(min(vl))}): ¿pusiste restore_best_weights=True en EarlyStopping?")
    # 3. Lo entregado frente a lo que da su propia red
    vals = _leer({"f1": f1, "sensibilidad": sensibilidad}, ["f1", "sensibilidad"])
    pred = (prob >= 0.5).astype(int)
    y = d["yr_p"]
    real = {"f1": f1_score(y, pred, zero_division=0), "sensibilidad": recall_score(y, pred, zero_division=0)}
    otros_val = {"exactitud": accuracy_score(y, pred), "precisión": precision_score(y, pred, zero_division=0)}
    pred_v = (prob_v >= 0.5).astype(int)
    ok = {}
    for nombre, v in vals.items():
        pista = None
        for otro, valor in [*otros_val.items(), *[(k, real[k]) for k in real if k != nombre]]:
            if not _cerca(v, real[nombre], 0.001) and _cerca(v, valor, 0.001):
                articulo = {"exactitud": "la exactitud", "precisión": "la precisión", "f1": "el F1",
                            "sensibilidad": "la sensibilidad"}[otro]
                pista = f"ese valor es {articulo} de tu red; aquí va {'el F1' if nombre == 'f1' else 'la sensibilidad'}"
                break
        if pista is None and not _cerca(v, real[nombre], 0.001) and _cerca(
                v, (f1_score if nombre == "f1" else recall_score)(d["yr_v"], pred_v, zero_division=0), 0.001):
            pista = "lo mediste con la validación; aquí va la prueba (yc_prueba y pred_red)"
        ok[nombre] = pista is None and _cerca(v, real[nombre], 0.006)
        if ok[nombre]:
            print(f"✓  {nombre}: coincide con tu red")
            continue
        print(f"✗  {nombre}: {pista or 'no coincide con lo que da tu red en la prueba: calcula con yc_prueba y pred_red'}")
    # 4. ¿Está la red donde se espera?
    f1_real, s_real = real["f1"], real["sensibilidad"]
    dentro = (_RANGO_F1_RED[0] <= f1_real <= _RANGO_F1_RED[1]
              and _RANGO_SENS_RED[0] <= s_real <= _RANGO_SENS_RED[1])
    if not dentro:
        print(f"·  Tu red tiene en la prueba un F1 de {_coma(f1_real)} y una sensibilidad de {_coma(s_real)}, fuera de "
              f"lo que esperamos (F1 entre {_coma(_RANGO_F1_RED[0], 2)} y {_coma(_RANGO_F1_RED[1], 2)}). Las redes de Keras "
              "cambian un poco de un equipo a otro: vuelve a ejecutar la celda; si sigue igual, prueba una tasa de dropout "
              "entre 0.2 y 0.5 (con punto, como en el código) y una paciencia entre 3 y 15")
    elif all(ok.get(k) for k in ("f1", "sensibilidad")) and tasas and 0 < max(tasas) < 0.8 and restaurada:
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
        if "llamadas_log" not in _CACHE:
            log = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(d["Xc_e"], d["yc_e"])
            _CACHE["llamadas_log"] = _llamadas_para(log.predict_proba(d["Xc_p"])[:, 1], y)
        llamadas = _llamadas_para(prob, y)
        print(f"\n¡Listo! Tu red {'paró en' if paro else 'corrió hasta'} la época {n} (la mejor fue la {mejor + 1}) y volvió a esos pesos. En la prueba: "
              f"F1 de {_coma(f1_real)} y sensibilidad de {_coma(s_real)}. Para encontrar a 7 de cada 10 necesita "
              f"{llamadas} llamadas; la logística, {_CACHE['llamadas_log']}. Compárala en «Tus respuestas» con la "
              "tabla de la celda [comparacion].")


# ---------------------------------------------------------------------------------------------
# Ejercicio 3 · Correr un píxel (Keras: se usan las redes del estudiante, sin reentrenar)
# ---------------------------------------------------------------------------------------------

def _correr(imagenes, pixeles=1):
    import numpy as np
    corridas = np.zeros_like(imagenes)
    corridas[:, :, pixeles:] = imagenes[:, :, :imagenes.shape[2] - pixeles]
    return corridas


def _exactitud(modelo, imagenes, etiquetas):
    import numpy as np
    return float((np.asarray(modelo.predict(imagenes, verbose=0)).argmax(axis=1) == etiquetas).mean())


@_seguro
def revisar_corrimiento(*sueltos, cnn=..., densa=..., exactitud_cnn=..., exactitud_densa=...,
                        corrida_cnn=..., corrida_densa=..., **otros):
    """Ejercicio 3: la exactitud de la CNN y de la densa, con las imágenes como vienen y corridas un píxel."""
    nombres = ["cnn", "densa", "exactitud_cnn", "exactitud_densa", "corrida_cnn", "corrida_densa"]
    if _sobrantes(sueltos, otros, nombres, "revisar_corrimiento(cnn=cnn, densa=densa, exactitud_cnn=…, "
                                           "exactitud_densa=…, corrida_cnn=…, corrida_densa=…)"):
        return
    for nombre, x in (("cnn", cnn), ("densa", densa)):
        if not _es_red(x):
            print(f"✗  {nombre}: pasa la red de Keras de la clase: {nombre}={nombre}")
            return
    if "Conv2D" not in _capas(cnn) and "Conv2D" in _capas(densa):
        print("✗  cnn y densa están cruzadas: la CNN es la que tiene capas Conv2D")
        return
    if "Conv2D" not in _capas(cnn):
        print("✗  cnn: esa red no tiene capas Conv2D; pasa la CNN de la celda [cnn]")
        return
    if "Conv2D" in _capas(densa):
        print("✗  densa: esa red tiene capas Conv2D; pasa la red densa de la celda [cnn]")
        return
    d = _digitos()
    I_p, d_p, I_e, d_e = d["I_p"], d["d_p"], d["I_e"], d["d_e"]
    corr = _correr(I_p)
    real = {"exactitud_cnn": _exactitud(cnn, I_p, d_p), "exactitud_densa": _exactitud(densa, I_p, d_p),
            "corrida_cnn": _exactitud(cnn, corr, d_p), "corrida_densa": _exactitud(densa, corr, d_p)}
    alternativas = {
        "con las imágenes corridas": {"exactitud_cnn": real["corrida_cnn"], "exactitud_densa": real["corrida_densa"]},
        "con las imágenes como vienen (I_prueba)": {"corrida_cnn": real["exactitud_cnn"],
                                                    "corrida_densa": real["exactitud_densa"]},
        "con las imágenes del entrenamiento": {"exactitud_cnn": _exactitud(cnn, I_e, d_e),
                                               "exactitud_densa": _exactitud(densa, I_e, d_e)},
        "con las imágenes corridas dos píxeles": {"corrida_cnn": _exactitud(cnn, _correr(I_p, 2), d_p),
                                                  "corrida_densa": _exactitud(densa, _correr(I_p, 2), d_p)},
    }
    vals = _leer({k: v for k, v in zip(nombres[2:], (exactitud_cnn, exactitud_densa, corrida_cnn,
                                                      corrida_densa))}, nombres[2:])
    ok = {}
    for nombre, v in vals.items():
        pista = None
        for como, valores in alternativas.items():
            if nombre in valores and _cerca(v, valores[nombre], 0.001) and not _cerca(v, real[nombre], 0.001):
                pista = f"ese valor sale de medir {como}"
                break
        if pista is None and not _cerca(v, real[nombre], 0.001):
            cruzado = nombre.replace("cnn", "X").replace("densa", "cnn").replace("X", "densa")
            if _cerca(v, real[cruzado], 0.001):
                pista = "ese valor es el de la otra red"
        ok[nombre] = pista is None and _cerca(v, real[nombre], 0.006)
        if ok[nombre]:
            print(f"✓  {nombre}: coincide con tu red")
            continue
        print(f"✗  {nombre}: {pista or 'no coincide con lo que dan tus redes: usa exactitud(red, imágenes)'}")
    if len(ok) == 4 and all(ok.values()):
        ec, ed, cc, cd = (real[k] for k in nombres[2:])
        if min(ec, ed) < _MINIMO_NORMAL or cc - cd <= _MARGEN_CORRIDA:
            print(f"·  Tus cifras están bien calculadas, pero tus redes salieron distintas de lo esperado (CNN "
                  f"{_coma(ec)} y {_coma(cc)}; densa {_coma(ed)} y {_coma(cd)}). Vuelve a ejecutar las celdas "
                  "[cnn] y [exactitud] y repite")
            return
        print(f"\n¡Listo! Con las imágenes como vienen, la CNN tiene una exactitud de {_coma(ec)} y la densa, de {_coma(ed)}. "
              f"Corridas un píxel, la CNN conserva {_coma(cc)} y la densa cae a {_coma(cd)}. Explica en «Tus respuestas» qué "
              "parte de la CNN la ayuda.")

# Descarga datos/clientes.csv: los mismos 8000 clientes de las sesiones 1 a 3
ruta = descargar_datos("datos")

## 1 · Los datos y la partición de la sesión 2

Volvemos a la pregunta de la gerente de experiencia del cliente: *¿quién va a cancelar?* Preparamos los
datos exactamente como en la sesión 2: cada categoría en columnas de 0 y 1, la etiqueta `cancelo` en 1 o 0,
y la cuarta parte de los clientes guardada para la **prueba**, con la semilla 2026 y la misma tasa de
cancelación (`stratify`). Así, los 2000 clientes de prueba son los mismos de la sesión 2 y podemos comparar.

In [ ]:
# [preparar]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

clientes = pd.read_csv("datos/clientes.csv")
Xc = pd.get_dummies(clientes.drop(columns=["id_cliente", "cancelo"]),
                    drop_first=True, dtype=int)
yc = (clientes["cancelo"] == "Sí").astype(int)   # 1 = canceló
Xc_ent, Xc_prueba, yc_ent, yc_prueba = train_test_split(
    Xc, yc, test_size=0.25, random_state=2026, stratify=yc)
print(f"{Xc.shape[1]} columnas · prueba: {len(yc_prueba)} clientes;",
      f"cancelaron {yc_prueba.sum()}")

---
# Parte 1 · De la neurona a la red

## 2 · Una neurona es una regresión logística

Una **neurona** hace cuatro cosas: multiplica cada entrada por un **peso**, suma, agrega un **sesgo** (*bias*) y
pasa el resultado por una **función de activación**. Con la activación **sigmoide**, $\sigma(z) = 1/(1 + e^{-z})$,
el resultado queda entre 0 y 1: es la regresión logística de la sesión 2. Lo comprobamos con un cliente:
calculamos a mano la suma ponderada y la sigmoide, y la comparamos con `predict_proba`.

In [ ]:
# [neurona]
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
logistica = make_pipeline(StandardScaler(),
                          LogisticRegression(max_iter=1000))
logistica.fit(Xc_ent, yc_ent)                  # la de la sesión 2
escalador, neurona = logistica[0], logistica[1]
x = escalador.transform(Xc_prueba.iloc[[0]])   # un cliente, escalado
z = x @ neurona.coef_.ravel() + neurona.intercept_[0]  # pesos y sesgo
p = 1 / (1 + np.exp(-z[0]))                    # activación sigmoide
print(f"{neurona.coef_.size} pesos y 1 sesgo · z = {z[0]:.3f}",
      f"→ P(cancela) = {p:.3f}")
p_sk = logistica.predict_proba(Xc_prueba.iloc[[0]])[0, 1]
print(f"predict_proba: {p_sk:.3f}")

## 3 · Lo que una sola neurona no puede

En la sesión 2, la regresión lineal no siguió el consumo de los clientes: el consumo baja con la edad, y baja
más rápido en los hogares con muchas personas (la **interacción** edad × usuarios). La regresión lineal es una
neurona **sin activación**: solo suma. Una **capa oculta** pone varias neuronas entre las columnas y la
salida; cada una aprende su propia combinación, y la activación ReLU deja que la red se doble. Con el
consumo (`consumo_gb`) como etiqueta y la partición de la regresión de la sesión 2:

In [ ]:
# [capa_oculta]
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
predictoras = clientes.drop(columns=["id_cliente", "consumo_gb", "cancelo"])
X = pd.get_dummies(predictoras, drop_first=True, dtype=int)
y = clientes["consumo_gb"]
X_ent, X_prueba, y_ent, y_prueba = train_test_split(
    X, y, test_size=0.25, random_state=2026)
recta = LinearRegression().fit(X_ent, y_ent)  # una neurona, sin activar
capa = make_pipeline(StandardScaler(), MLPRegressor(
    hidden_layer_sizes=(32,), learning_rate_init=0.01,
    early_stopping=True, random_state=2026))   # 32 neuronas ocultas
capa.fit(X_ent, y_ent)
for nombre, modelo in {"Una neurona (la recta)": recta,
                       "Una capa oculta de 32": capa}.items():
    mae = mean_absolute_error(y_prueba, modelo.predict(X_prueba))
    print(f"{nombre}: MAE {mae:.1f} GB")

## 4 · Las activaciones

Sin activación, apilar capas no sirve: una suma de sumas sigue siendo una suma (una recta). La activación
dobla la línea. Dos son las de esta parte (la **softmax** llega con los dígitos):

- **ReLU**, $\max(0, z)$: deja pasar lo positivo y apaga lo negativo. Es la opción por defecto en las capas
  ocultas.
- **Sigmoide**: lleva cualquier número a una probabilidad entre 0 y 1. Va en la salida cuando la pregunta
  es de sí o no.

In [ ]:
# [activaciones]
z = np.linspace(-6, 6, 200)
fig, ejes = plt.subplots(1, 2, figsize=(8.4, 3.2))
ejes[0].plot(z, np.maximum(0, z), color="#8A2E9E", linewidth=2.5)
ejes[0].set_title("ReLU: max(0, z)")
ejes[1].plot(z, 1 / (1 + np.exp(-z)), color="#F46201", linewidth=2.5)
ejes[1].set_title("Sigmoide: entre 0 y 1")
for eje in ejes:
    eje.axhline(0, color="#94A3B8", linewidth=0.8)
    eje.axvline(0, color="#94A3B8", linewidth=0.8)
    eje.set_xlabel("z, la suma ponderada más el sesgo")
plt.show()

## 5 · Entrenar es bajar la pérdida, paso a paso

La **función de pérdida** dice cuánto se equivoca la red. Para un sí o un no se usa la **entropía cruzada
binaria**: $-[\,y \log p + (1-y)\log(1-p)\,]$, donde $p$ es la probabilidad de cancelar que da la red; castiga
mucho una probabilidad de cancelar alta para quien no canceló (o baja para quien sí). Entrenar es mover los pesos para que la pérdida baje: el **descenso del gradiente**
mira hacia dónde baja la pérdida (la pendiente) y da un paso en esa dirección. El tamaño del paso es la
**tasa de aprendizaje**. La gráfica lo muestra con un solo peso y una pérdida en forma de U.

In [ ]:
# [descenso]
def bajar(tasa, inicio=-1.0, pasos=12):
    """Descenso del gradiente en la pérdida (w − 3)² + 1."""
    w, camino = inicio, [inicio]
    for _ in range(pasos):
        w = w - tasa * 2 * (w - 3)         # la pendiente es 2(w − 3)
        camino.append(w)
    return np.array(camino)

ws = np.linspace(-2, 8, 200)
fig, ejes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True)
for eje, (tasa, nombre) in zip(ejes, [(0.03, "Paso corto: lento"),
                                      (0.3, "Paso bueno"),
                                      (0.97, "Paso largo: rebota")]):
    camino = bajar(tasa)
    eje.plot(ws, (ws - 3) ** 2 + 1, color="#94A3B8", linewidth=2)
    eje.plot(camino, (camino - 3) ** 2 + 1, "o-", color="#8A2E9E",
             markersize=5, linewidth=1.2)
    eje.set_title(f"{nombre} ({str(tasa).replace('.', ',')})")
    eje.set_xlabel("Peso w")
ejes[0].set_ylabel("Pérdida")
plt.show()

Con un paso corto, la red avanza poco y tarda en llegar; con uno largo, salta de un lado al otro del valle. Una red
tiene miles de pesos, y la **retropropagación** (*backpropagation*) calcula la pendiente de la pérdida para
todos a la vez. Tres palabras más:

- una **época** es una pasada completa por los clientes del entrenamiento;
- un **lote** (*batch*) es el grupo de clientes con que se calcula cada paso (aquí, 32);
- **Adam** es el optimizador que usamos: ajusta por separado el tamaño del paso de cada peso.

---
# Parte 2 · La red en Yarumo

## 6 · Una red de scikit-learn: `MLPClassifier`

`MLPClassifier` es un **perceptrón multicapa** (MLP): capas ocultas con ReLU y una salida con la
probabilidad de cancelar. Como KNN y la SVM de la sesión 2, es sensible a la escala de las columnas, así que
va en un `Pipeline` que primero escala. `early_stopping=True` aparta el 10 % del entrenamiento y para
cuando la red deja de mejorar en esos clientes.

In [ ]:
# [mlp]
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.neural_network import MLPClassifier
mlp = make_pipeline(StandardScaler(), MLPClassifier(
    hidden_layer_sizes=(32,), early_stopping=True, random_state=2026))
mlp.fit(Xc_ent, yc_ent)
pred_mlp = mlp.predict(Xc_prueba)
print(f"Precisión {precision_score(yc_prueba, pred_mlp):.3f}",
      f"· sensibilidad {recall_score(yc_prueba, pred_mlp):.3f}",
      f"· F1 {f1_score(yc_prueba, pred_mlp):.3f}")

## 7 · La misma idea en Keras

**Keras** arma una red capa por capa y la entrena con TensorFlow por debajo. El *backend* (el motor que hace las
cuentas: TensorFlow, JAX o PyTorch) se fija **antes** de importar Keras. Además, en cada red se fija la semilla antes de armarla,
para que los pesos al azar del comienzo sean los mismos en cada ejecución.

Para ver si la red memoriza hace falta un tercer grupo de clientes, la **validación**: sale del
entrenamiento (la prueba sigue guardada para el final) y la red no aprende de él; solo lo mira al final de
cada época. La partimos a mano, estratificada, y el escalador aprende solo de los clientes con que se entrena.

In [ ]:
# [keras]
import os
os.environ["KERAS_BACKEND"] = "tensorflow"   # antes de importar Keras
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"     # menos avisos técnicos
import keras
from keras import layers

Xc_ent2, Xc_val, yc_ent2, yc_val = train_test_split(
    Xc_ent, yc_ent, test_size=0.2, random_state=2026, stratify=yc_ent)
escala = StandardScaler().fit(Xc_ent2)   # solo del entrenamiento
Xr_ent = escala.transform(Xc_ent2).astype("float32")
Xr_val = escala.transform(Xc_val).astype("float32")
Xr_prueba = escala.transform(Xc_prueba).astype("float32")
yr_ent, yr_val = yc_ent2.to_numpy(), yc_val.to_numpy()
print(f"Keras {keras.__version__} · entrenamiento {len(yr_ent)}",
      f"· validación {len(yr_val)} · prueba {len(Xr_prueba)}")

## 8 · Una red grande, sin regularizar

Dos capas ocultas de 256 neuronas para 4800 clientes: muchos más pesos que clientes. La entrenamos 40 épocas
y guardamos el historial (`h_grande`): la pérdida en el entrenamiento y en la validación, época por época.

In [ ]:
# [red_grande]
keras.utils.set_random_seed(2026)            # misma semilla, mismos pesos
grande = keras.Sequential([
    keras.Input(shape=(Xr_ent.shape[1],)),   # una entrada por columna
    layers.Dense(256, activation="relu"),    # capa oculta 1
    layers.Dense(256, activation="relu"),    # capa oculta 2
    layers.Dense(1, activation="sigmoid"),   # salida: P(cancela)
])
grande.compile(optimizer="adam", loss="binary_crossentropy")
h_grande = grande.fit(Xr_ent, yr_ent, validation_data=(Xr_val, yr_val),
                      epochs=40, batch_size=32, verbose=0)
print(f"{grande.count_params()} pesos y sesgos para {len(yr_ent)} clientes")

In [ ]:
# [curva_perdida]
perdida = h_grande.history["loss"]
perdida_val = h_grande.history["val_loss"]
epocas = np.arange(1, len(perdida) + 1)
mejor = int(np.argmin(perdida_val)) + 1      # la época con menos pérdida
fig, ax = plt.subplots(figsize=(8.4, 3.9))
ax.plot(epocas, perdida, color="#8A2E9E", linewidth=2.5,
        label="Entrenamiento")
ax.plot(epocas, perdida_val, color="#F46201", linewidth=2.5,
        label="Validación")
ax.axvline(mejor, color="#566B7F", linestyle="--", linewidth=1)
ax.set_title("La red grande memoriza: la validación sube casi desde el comienzo")
ax.set_xlabel("Época")
ax.set_ylabel("Pérdida (entropía cruzada)")
ax.legend(frameon=False)
plt.show()

En el entrenamiento, la pérdida baja casi hasta cero: la red memoriza a los 4800 clientes. En la validación
toca fondo en las primeras épocas (la línea punteada) y después sube: con clientes que no vio, cada época la
hace peor. Es el sobreajuste de la sesión 2, ahora época por época.

## 9 · La comparación honesta, con las métricas de la sesión 2

La gerente no decide con la exactitud. Comparamos con lo que ya sabemos leer: la **precisión**, la
**sensibilidad** y el **F1** con el umbral de 0,5. También contamos las **llamadas** que hacen falta para
encontrar a 7 de cada 10 de los que cancelan, ordenando a los clientes de mayor a menor riesgo (la función de
la sesión 2).
Entrenamos también XGBoost, como en la sesión 2.

In [ ]:
# [comparacion]
from xgboost import XGBClassifier
xgb_c = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3,
                      random_state=2026).fit(Xc_ent, yc_ent)

def llamadas_para(riesgo, meta=0.70):
    """Llamadas, de mayor a menor riesgo, para llegar a la meta."""
    y_real = yc_prueba.to_numpy()
    for corte in np.sort(np.unique(riesgo))[::-1]:
        llamados = riesgo >= corte          # los empatados, todos juntos
        if y_real[llamados].sum() >= meta * y_real.sum():
            return int(llamados.sum())

def metricas(riesgo):
    """Las métricas de la sesión 2 para un riesgo por cliente."""
    pred = (riesgo >= 0.5).astype(int)      # el umbral de 0.5
    return {"precisión": precision_score(yc_prueba, pred),
            "sensibilidad": recall_score(yc_prueba, pred),
            "F1": f1_score(yc_prueba, pred),
            "llamadas 7 de 10": llamadas_para(riesgo)}

riesgos = {"Logística": logistica.predict_proba(Xc_prueba)[:, 1],
           "MLP (scikit-learn)": mlp.predict_proba(Xc_prueba)[:, 1],
           "Red grande (Keras)": grande.predict(Xr_prueba,
                                                verbose=0).ravel(),
           "XGBoost": xgb_c.predict_proba(Xc_prueba)[:, 1]}
tabla = pd.DataFrame({m: metricas(r) for m, r in riesgos.items()}).T
tabla.astype({"llamadas 7 de 10": int}).round(3)

El MLP iguala a la logística; la red grande que memorizó necesita más llamadas que las dos; XGBoost sigue al
frente. En una tabla de clientes, con columnas que ya dicen mucho, los árboles son una línea base muy
difícil de superar. Las redes ganan cuando el dato es una **imagen**, un **texto** o una **secuencia**:
la parte 3.

## 10 · Regularizar: que la red no memorice

Tres formas de frenar a una red que memoriza:

- **Parada temprana** (*early stopping*): mira la pérdida de validación al final de cada época y para cuando
  lleva `patience` épocas sin mejorar; con `restore_best_weights=True`, la red vuelve a los pesos de su mejor
  época. En Keras es un *callback* (una función que Keras llama durante el entrenamiento, al final de cada época),
  `keras.callbacks.EarlyStopping`.
- ***Dropout***: en cada paso del entrenamiento apaga al azar una fracción de las neuronas (`rate`, entre 0 y
  1), y así ninguna puede depender de otra en particular. Al predecir, todas trabajan. En Keras es una capa,
  `layers.Dropout(rate)`, después de la capa que quieres frenar.
- **L2**: castiga los pesos grandes. En `MLPClassifier` es el parámetro `alpha`.

---
## Ejercicio 1 · Escalar importa

Entrena un MLP de **dos capas ocultas, de 16 y 8 neuronas**, dos veces: sin escalar y con escalado (en un
`Pipeline` que primero escala), y calcula el **F1 en prueba** de cada uno. El código está casi completo:
reemplaza cada `...`. Pista: la celda [mlp]. `zero_division=0` dice qué hacer si el modelo no señala a nadie:
en ese caso, el F1 es 0.

Después responde en «Tus respuestas», al final del notebook: ¿por qué cambia tanto? ¿Qué modelo de la sesión 2
también dijo «No» a todos sin escalar?

In [ ]:
# [ej1]
mlp_sin = MLPClassifier(hidden_layer_sizes=(16, 8), early_stopping=True,
                        random_state=2026)            # sin escalar
mlp_con = make_pipeline(..., MLPClassifier(
    hidden_layer_sizes=(16, 8), early_stopping=True,
    random_state=2026))                               # primero escala
mlp_sin.fit(Xc_ent, yc_ent)
mlp_con.fit(..., ...)                       # entrena con el entrenamiento
f1_sin = f1_score(yc_prueba, mlp_sin.predict(Xc_prueba), zero_division=0)
f1_con = ...                                # el F1 en prueba del escalado
revisar_escalado(f1_sin=f1_sin, f1_con=f1_con)

## Ejercicio 2 · Que no memorice

La red es la misma de la celda [red_grande]: dos capas ocultas de 256. Le agregamos un *dropout* después de
cada capa oculta y la parada temprana. Completa dos valores:

- `tasa_dropout`: la tasa de *dropout*, la fracción de neuronas que se apagan en cada paso (un número entre 0 y 1;
  prueba entre 0.2 y 0.5). No es la tasa de aprendizaje;
- `paciencia`: cuántas épocas seguidas sin mejorar en la validación espera antes de parar (prueba entre 3 y 15).

Después calcula el **F1** y la **sensibilidad** en la prueba, con el umbral de 0,5. Pista: las celdas [mlp] y
[comparacion]. El verificador revisa tus cifras con tu propia red y mira su historial.

Cuando el verificador diga «¡Listo!», ejecuta la celda otra vez con `tasa_dropout = 0` y después con `0.8` (la
misma paciencia), y anota en «Tus respuestas» la época en que paró, su mejor época y el F1 de cada una. El
verificador te dirá que esas dos no son la respuesta: es lo esperado. ¿Cuánto aporta el *dropout*? ¿Qué relación
hay entre la época en que paró, la mejor y la paciencia?

In [ ]:
# [ej2]
tasa_dropout = ...         # entre 0 y 1: la fracción que se apaga
paciencia = ...            # épocas sin mejorar antes de parar
keras.utils.set_random_seed(2026)
red = keras.Sequential([
    keras.Input(shape=(Xr_ent.shape[1],)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(tasa_dropout),            # apaga neuronas al azar
    layers.Dense(256, activation="relu"),
    layers.Dropout(tasa_dropout),
    layers.Dense(1, activation="sigmoid"),
])
red.compile(optimizer="adam", loss="binary_crossentropy")
parada = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=paciencia, restore_best_weights=True)
h = red.fit(Xr_ent, yr_ent, validation_data=(Xr_val, yr_val), epochs=100,
            batch_size=32, verbose=0, callbacks=[parada])
pred_red = (red.predict(Xr_prueba, verbose=0).ravel() >= 0.5).astype(int)
f1_red = ...               # el F1 en prueba
sens_red = ...             # la sensibilidad en prueba
revisar_red(red=red, historial=h, f1=f1_red, sensibilidad=sens_red)

---
# Parte 3 · Imágenes y secuencias

Si no terminaste el ejercicio 1 o el 2, ejecuta estas celdas una por una con **Mayús + Enter**: no dependen de
los ejercicios.

## 11 · Dígitos escritos a mano, de 8 × 8 píxeles

En la tabla de clientes, la red no le ganó a XGBoost. Donde las redes brillan es en datos con **forma**: una
imagen, donde cada píxel tiene vecinos. scikit-learn trae 1797 dígitos escritos a mano (`load_digits`), de
8 × 8 píxeles con valores de 0 a 16. No hay que descargar nada. Los dividimos entre 16 para que vayan de
0 a 1, y les agregamos un **canal** (la última dimensión), como espera Keras.

In [ ]:
# [digitos]
from sklearn.datasets import load_digits
digitos = load_digits()
# de 0 a 1, y con un canal: (imágenes, alto, ancho, 1)
imagenes = (digitos.images / 16).reshape(-1, 8, 8, 1).astype("float32")
I_ent, I_prueba, d_ent, d_prueba = train_test_split(
    imagenes, digitos.target, test_size=0.25, random_state=2026,
    stratify=digitos.target)
print(imagenes.shape, "→", len(I_ent), "para entrenar y",
      len(I_prueba), "para probar")

In [ ]:
# [digitos_figura]
fig, ejes = plt.subplots(1, 10, figsize=(10.5, 1.6))
for k, eje in enumerate(ejes):
    eje.imshow(digitos.images[k], cmap="Purples")
    eje.set_title(str(digitos.target[k]))
    eje.axis("off")
plt.show()

## 12 · Un filtro de 3 × 3 que recorre la imagen

Una **convolución** pasa un **filtro**, una ventana de 3 × 3 pesos, por cada posición de la imagen: en cada
una multiplica los 9 píxeles por los 9 pesos y suma. Los **mismos 9 pesos** sirven en toda la imagen. Este
filtro, hecho a mano, responde a los bordes verticales: positivo donde la tinta aparece de izquierda a derecha
y negativo donde desaparece. Una CNN no recibe los filtros hechos: los aprende.

In [ ]:
# [filtro]
filtro = np.array([[-1, 0, 1],
                   [-1, 0, 1],
                   [-1, 0, 1]])              # bordes verticales
imagen = digitos.images[0]                   # un 0
mapa = np.zeros((6, 6))
for i in range(6):                           # cada posición de la ventana
    for j in range(6):
        mapa[i, j] = (imagen[i:i + 3, j:j + 3] * filtro).sum()
fig, ejes = plt.subplots(1, 2, figsize=(6.4, 3.0))
ejes[0].imshow(imagen, cmap="Purples")
ejes[0].set_title("La imagen (8 × 8)")
ejes[1].imshow(mapa, cmap="PuOr_r")
ejes[1].set_title("Con el filtro (6 × 6)")
for eje in ejes:
    eje.axis("off")
plt.show()

La ventana de 3 × 3 cabe en 6 × 6 posiciones de una imagen de 8 × 8. Con `padding="same"`, Keras rodea la
imagen de ceros para que el mapa conserve su tamaño de 8 × 8. Después, el ***pooling*** resume: `MaxPooling2D(2)`
se queda con el máximo de cada cuadro de 2 × 2 (8 × 8 pasa a 4 × 4), y `GlobalMaxPooling2D` se queda con el
máximo de cada filtro en toda la imagen: dice *si* apareció el rasgo, no *dónde*.

## 13 · Una CNN y una red densa, con casi los mismos pesos

La **red convolucional** (CNN) tiene dos capas de filtros, *pooling* y una salida **softmax**: diez
probabilidades, una por dígito, que suman 1. La **red densa** aplana la imagen en 64 números y la trata como
una fila de una tabla. Con diez clases, la pérdida es la entropía cruzada para varias clases.

In [ ]:
# [cnn]
keras.utils.set_random_seed(2026)
cnn = keras.Sequential([
    keras.Input(shape=(8, 8, 1)),
    layers.Conv2D(16, 3, padding="same", activation="relu"),  # 16 filtros
    layers.MaxPooling2D(2),                   # 8 × 8 → 4 × 4
    layers.Conv2D(32, 3, padding="same", activation="relu"),  # 32 filtros
    layers.GlobalMaxPooling2D(),              # el máximo de cada filtro
    layers.Dense(10, activation="softmax"),   # P de cada dígito
])
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
h_cnn = cnn.fit(I_ent, d_ent, epochs=30, batch_size=32, verbose=0)

keras.utils.set_random_seed(2026)
densa = keras.Sequential([
    keras.Input(shape=(8, 8, 1)),
    layers.Flatten(),                         # 8 × 8 → 64 números
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
densa.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
h_densa = densa.fit(I_ent, d_ent, epochs=30, batch_size=32, verbose=0)

In [ ]:
# [exactitud]
def exactitud(modelo, imagenes, etiquetas=d_prueba):
    """La proporción de dígitos que acierta (el de mayor probabilidad)."""
    prob = modelo.predict(imagenes, verbose=0)   # 10 probabilidades
    return float((prob.argmax(axis=1) == etiquetas).mean())

for nombre, modelo in {"CNN": cnn, "Densa": densa}.items():
    print(f"{nombre}: {modelo.count_params()} pesos ·",
          f"exactitud {exactitud(modelo, I_prueba):.3f}")

Con las imágenes tal como vienen, las dos aciertan casi todo. Pero en una foto real el dígito no siempre
está centrado. ¿Qué pasa si cada imagen se **corre un píxel** a la derecha? La función `correr` lo hace: mueve
las columnas y rellena con ceros la que queda vacía.

In [ ]:
# [correr]
def correr(imagenes, pixeles=1):
    """Corre cada imagen hacia la derecha; entran columnas de ceros."""
    corridas = np.zeros_like(imagenes)
    corridas[:, :, pixeles:] = imagenes[:, :, :imagenes.shape[2] - pixeles]
    return corridas

I_corridas = correr(I_prueba)
fig, ejes = plt.subplots(1, 2, figsize=(4.6, 2.6))
ejes[0].imshow(I_prueba[0, :, :, 0], cmap="Purples")
ejes[0].set_title("Original")
ejes[1].imshow(I_corridas[0, :, :, 0], cmap="Purples")
ejes[1].set_title("Un píxel a la derecha")
for eje in ejes:
    eje.axis("off")
plt.show()

## Ejercicio 3 · Correr un píxel

Si el ejercicio 2 quedó a medias: haz clic en la celda [digitos] → **Entorno de ejecución → Ejecutar después**.
Se detiene aquí, en el ejercicio 3.

Antes de ejecutar, predice y anota en «Tus respuestas»: con las imágenes corridas, ¿cuál pierde más, la CNN o la
densa?

Calcula la exactitud de las dos redes con las imágenes de prueba **tal como vienen** (`I_prueba`) y con las
**corridas** (`I_corridas`). Pista: la función `exactitud` de la celda [exactitud].

Después responde en «Tus respuestas»: ¿cuál conserva más? ¿Qué parte de la CNN crees que la ayuda? Si te sobra tiempo,
prueba con `correr(I_prueba, 2)`.

In [ ]:
# [ej3]
exactitud_cnn = ...              # la CNN, con I_prueba
exactitud_densa = ...            # la densa, con I_prueba
corrida_cnn = ...                # la CNN, con I_corridas
corrida_densa = ...              # la densa, con I_corridas
revisar_corrimiento(cnn=cnn, densa=densa,
                    exactitud_cnn=exactitud_cnn,
                    exactitud_densa=exactitud_densa,
                    corrida_cnn=corrida_cnn, corrida_densa=corrida_densa)

## 14 · Una secuencia: el tráfico de Yarumo, hora por hora

El jefe de TI quiere anticipar el tráfico de la próxima hora para planear la capacidad. Yarumo no nos dio esa
serie, así que la **inventamos** aquí, con una fórmula a la vista (es ficticia, como todo el caso): 70 días,
hora por hora, en Gbps (gigabits por segundo), con

- un ciclo diario: más tráfico hacia las 5 p. m. y menos hacia las 5 a. m.;
- 8 Gbps más los fines de semana (el día 0 es un lunes);
- un ruido que arrastra el 70 % del de la hora anterior: si una hora vino alta, la siguiente suele venir alta.

In [ ]:
# [serie]
rng = np.random.default_rng(2026)
horas = np.arange(24 * 70)                   # 70 días, hora por hora
hora, dia = horas % 24, horas // 24
ruido = np.zeros(len(horas))
for t in range(1, len(horas)):          # arrastra el de la hora anterior
    ruido[t] = 0.7 * ruido[t - 1] + rng.normal(0, 3)
trafico = (40 + 25 * np.sin(2 * np.pi * (hora - 11) / 24)  # ciclo diario
           + 8 * (dia % 7 >= 5)              # fin de semana
           + ruido)                           # Gbps
print(f"{len(trafico)} horas · de {trafico.min():.1f} a",
      f"{trafico.max():.1f} Gbps")

In [ ]:
# [serie_grafica]
fig, ax = plt.subplots(figsize=(8.4, 3.2))
ax.plot(horas[:24 * 14] / 24, trafico[:24 * 14], color="#8A2E9E",
        linewidth=1.5)
ax.set_title("Dos semanas de tráfico (ficticio)")
ax.set_xlabel("Día (el 0 es un lunes)")
ax.set_ylabel("Gbps")
plt.show()

Para predecir la hora siguiente, cada ejemplo es una **ventana** con las últimas 24 horas, y la etiqueta es la
hora que sigue. En una serie de tiempo **no se baraja**: se entrena con el pasado (el primer 80 %) y se prueba
con el futuro (el último 20 %, unos 14 días).

In [ ]:
# [ventanas]
V = 24                                       # la ventana: 24 horas
X_s = np.array([trafico[t - V:t] for t in range(V, len(trafico))])
y_s = trafico[V:]                            # la hora siguiente
corte = int(len(y_s) * 0.8)                  # sin barajar
Xs_ent, Xs_prueba = X_s[:corte], X_s[corte:]
ys_ent, ys_prueba = y_s[:corte], y_s[corte:]
print(f"{len(ys_ent)} ventanas para entrenar y {len(ys_prueba)}",
      "para probar")

Tres líneas base antes de la red: decir que la próxima hora será igual a **la última hora**; igual a **la
misma hora de ayer**; o una **regresión lineal** con las 24 horas de la ventana. Medimos con el MAE de la
sesión 2: cuántos Gbps se equivoca, en promedio, cada predicción.

In [ ]:
# [bases_serie]
errores = {
    "La última hora": mean_absolute_error(ys_prueba, Xs_prueba[:, -1]),
    "La misma hora de ayer": mean_absolute_error(ys_prueba,
                                                 Xs_prueba[:, 0]),
}
recta_s = LinearRegression().fit(Xs_ent, ys_ent)   # pesos para 24 horas
errores["Regresión lineal (24 h)"] = mean_absolute_error(
    ys_prueba, recta_s.predict(Xs_prueba))
pd.DataFrame({"MAE en prueba (Gbps)": errores}).round(2)

## 15 · Una red recurrente

Una **red recurrente** (RNN) lee la ventana hora por hora y arrastra un **estado**: un resumen de lo que ya
leyó, que mezcla con cada hora nueva. `SimpleRNN` es la más sencilla. Como siempre, escalamos con el
entrenamiento.

In [ ]:
# [rnn]
media, desv = ys_ent.mean(), ys_ent.std()    # escala del entrenamiento
keras.utils.set_random_seed(2026)
rnn = keras.Sequential([
    keras.Input(shape=(V, 1)),               # 24 horas, un valor por hora
    layers.SimpleRNN(16),                    # lee y arrastra un estado
    layers.Dense(1),                         # la hora siguiente
])
rnn.compile(optimizer="adam", loss="mse")
rnn.fit(((Xs_ent - media) / desv)[..., None], (ys_ent - media) / desv,
        epochs=40, batch_size=32, verbose=0)
pred_rnn = rnn.predict(((Xs_prueba - media) / desv)[..., None],
                       verbose=0).ravel() * desv + media
errores["Red recurrente"] = mean_absolute_error(ys_prueba, pred_rnn)
pd.DataFrame({"MAE en prueba (Gbps)": errores}).round(2)

La red recurrente le gana a las dos primeras líneas base y **empata** con la regresión lineal. No es un fracaso
de la red: esta serie depende sobre todo de la última hora y de la hora del día, y eso una suma ponderada de
las 24 horas ya lo capta. La recurrencia sirve cuando lo que importa quedó lejos en la secuencia; para eso se
inventó la **LSTM**, una red recurrente con compuertas que deciden qué recordar y qué olvidar.

---
# Parte 4 · Nube: los datos de Yarumo en Azure

## Ejercicio de diseño · El plan de almacenamiento y monitoreo

Este ejercicio no lleva código: se hace en parejas, con las diapositivas de la sesión, y va directo a la parte
2 del proyecto. Para cada **activo** de Yarumo, completa las cinco columnas:

1. **Servicio y redundancia**: ¿un blob en Blob Storage o una tabla en Azure SQL Database? ¿LRS, ZRS o GRS?
2. **Quién lee**: el rol integrado y su ámbito (la matriz de roles de la sesión 2).
3. **Cómo se autoriza**: ¿Microsoft Entra ID, la clave compartida o una firma de acceso compartido (SAS)?
   ¿Acceso anónimo?
4. **Qué se registra y qué alerta**: ¿el registro de actividad, una configuración de diagnóstico con
   `StorageRead` hacia Log Analytics? ¿Qué alerta y a quién?
5. **Costo**: según la página oficial de servicios gratuitos de Azure, ¿cabe en lo gratuito? ¿Por cuánto tiempo?

*Doble clic para escribir aquí.*

| Activo | Servicio y redundancia | Quién lee (rol y ámbito) | Cómo se autoriza | Qué se registra y qué alerta | Costo |
|---|---|---|---|---|---|
Completa la fila de `clientes.csv` entera; en las otras dos, solo lo que cambia. Las tres van en Blob Storage, en
contenedores distintos (si prefieres una tabla de Azure SQL Database para el riesgo de cada cliente, dilo en la
columna 1). El riesgo de cada cliente es un dato personal: por eso va en un contenedor nuevo, `predicciones`, y no
en `informes`, que la gerencia lee sin clientes uno a uno.

| Activo | Servicio y redundancia | Quién lee (rol y ámbito) | Cómo se autoriza | Qué se registra y qué alerta | Costo |
|---|---|---|---|---|---|
| `clientes.csv` (contenedor `datos-clientes`; datos personales seudonimizados) | … | … | … | … | … |
| El modelo entrenado (contenedor `modelos`) | … | … | … | … | … |
| El riesgo de cada cliente (contenedor nuevo `predicciones`) | … | … | … | … | … |

---
### Tus respuestas

*Doble clic para escribir aquí.*

1. Ejercicio 1: sin escalar, el F1 fue de …; con escalado, de … ¿Por qué cambia tanto? ¿Qué modelo de la
   sesión 2 hizo lo mismo sin escalar? …
2. Ejercicio 2: con una tasa de *dropout* de … y una paciencia de …, la red paró en la época … (su mejor época
   fue la …) y tuvo un F1 de … y una sensibilidad de … Con tasa 0: paró en la …, mejor época …, F1 … Con 0,8:
   paró en la …, mejor época …, F1 … ¿Cuánto aporta el *dropout*? ¿Qué relación hay entre la época en que paró,
   la mejor y la paciencia? ¿Cómo queda frente a la red grande, al MLP, a la logística y a XGBoost (celda
   [comparacion])? ¿Le recomendarías a la gerente la red o XGBoost? …
3. Ejercicio 3: mi predicción fue … Con las imágenes corridas, la CNN acierta … y la densa, … ¿Qué parte de la
   CNN la ayuda? …
4. La red recurrente empató con la regresión lineal. ¿Con qué serie esperarías que le ganara? …

### Autoevaluación (respóndela en el cierre de la sesión)

¿Verdadero o falso? Explica cada una.

1. «Con la clave compartida desactivada y sin acceso anónimo, el Propietario de la suscripción lee
   `clientes.csv` aunque no tenga un rol de datos». …
2. «Con las imágenes corridas un píxel, la CNN conserva bastante más exactitud que una red densa de tamaño
   parecido». …
3. «Para saber quién leyó `clientes.csv` ayer, la configuración de diagnóstico tenía que existir antes de la
   lectura». …
4. «Con `restore_best_weights=True`, la parada temprana deja la red con los pesos de la última época». …

---
## La respuesta a la gerencia

**¿Una red neuronal encuentra mejor a los que van a cancelar?** No en esta tabla: el MLP iguala a la
logística y XGBoost sigue al frente; una red grande sin regularizar memoriza y queda peor. Con la parada
temprana y el *dropout*, la red mejora, pero no supera a XGBoost. Las redes valen cuando el dato es una
imagen, un texto o una secuencia.

**¿Dónde guardamos los datos en Azure, quién los lee y cómo nos enteramos?** En una cuenta de almacenamiento
sin acceso anónimo y sin la clave compartida, con roles de datos en el ámbito más estrecho. Las lecturas se
registran con una configuración de diagnóstico con `StorageRead` hacia Log Analytics, y una alerta avisa.
Defender for Cloud evalúa la configuración y da una puntuación de seguridad. El costo se calcula con la página
oficial de servicios gratuitos de Azure.